# 3d-point-cloud (SparseVoxFormer branch) -- Colab training

DETR-style set-prediction head, adapted from **SparseVoxFormer** (arXiv 2503.08092,
"Sparse Voxel-based Transformer for Multi-modal 3D Object Detection"). The paper's
core idea -- feed sparse voxel tokens straight into a transformer decoder instead of
first compressing them into a dense BEV map -- is already what this repo's trunk does
(`models/vfe.py` -> `models/backbone3d_auto.py` -> `models/slotformer.py`, unchanged
here). What's new on this branch is the part the paper adds on top and this repo
didn't have yet: a DETR-style query decoder + Hungarian bipartite matching +
set-prediction loss (`models/decoder_detr.py`, `models/matcher.py`,
`models/losses_detr.py`, `models/heads_detr.py`, `models/detector_detr.py`), as a
second head trained/evaluated side by side with the existing dense CenterPoint-style
head (`configs/default.yaml`) via the same config/eval conventions
(`configs/exp_detr_head.yaml`, `train_detr.py`, `test_detr.py`).

Rotation is 6D continuous representation (`models/rotation6d.py`), not this repo's
existing quaternion convention (`models/box_utils.py`) -- GT boxes stay quaternion in
the dataset; the DETR head's loss/matcher convert to a rotation matrix for comparison.

No camera/multi-modal fusion in this phase -- the dataset has no sonar-camera
calibration (no intrinsics/extrinsics, only a manual per-object `link_id` pairing),
so the paper's LiDAR-to-image voxel projection can't be honestly reproduced yet.

Runtime -> Change runtime type -> GPU, before running anything below.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Clone and install

The model code lives on the `SparseVoxFormer` branch specifically.

In [ ]:
import os

# Safe to re-run this cell any number of times, from any state -- avoids the
# nested-clone trap (3d-point-cloud/3d-point-cloud/...) that a plain
# `!git clone && %cd` gives you if you re-run this cell after the first `%cd`
# already moved you inside the repo.
if os.path.basename(os.getcwd()) == "3d-point-cloud" and os.path.exists("train_detr.py"):
    print("already inside 3d-point-cloud/ -- nothing to do")
else:
    if not os.path.isdir("3d-point-cloud"):
        !git clone -b SparseVoxFormer https://github.com/izione/3d-point-cloud.git
    else:
        print("3d-point-cloud/ already exists here -- skipping clone")
    %cd 3d-point-cloud

!pip install -q -r requirements.txt

Optional: spconv accelerates the sparse backbone if it installs cleanly for
this Colab image's CUDA version -- everything works without it too (pure-PyTorch
fallback, `models/backbone3d_auto.py` probes automatically and the DETR head has
been verified against both paths).

In [ ]:
# !pip install -q spconv-cu126   # pick the cuXXX tag matching this runtime's CUDA (see https://github.com/traveller59/spconv)

## 2. Get the dataset

Downloads `dataset.zip` from a Google Drive share link and unzips it onto the Colab
VM's local disk (`/content/dataset_extracted`) -- not the Drive-mounted path, since
`SonarDiverDataset` reads many small files per epoch and local disk is much faster
than Drive's network filesystem for that access pattern.

In [ ]:
DATASET_ZIP_SHARE_URL = "https://drive.google.com/file/d/1JTnVQ1c25z2MfJQUvIHz4dgyRsFiPLm_/view?usp=drive_link"

In [ ]:
!pip install -q gdown
!gdown --fuzzy "{DATASET_ZIP_SHARE_URL}" -O /content/dataset.zip
!unzip -q -o /content/dataset.zip -d /content/dataset_extracted
!ls /content/dataset_extracted

Auto-detect `DATASET_ROOT` (descends past wrapper folders until it finds
`Person*` folders) and patch `configs/default.yaml`'s `DATA.ROOT` to point at it --
`configs/exp_detr_head.yaml` inherits `DATA` from `default.yaml` (via `BASE_CONFIG`,
see `config_utils.py`), so patching the base file is enough.

In [ ]:
import os
import yaml

DATASET_ROOT = "/content/dataset_extracted"
while True:
    entries = [e for e in os.listdir(DATASET_ROOT) if not e.startswith(".")]
    if any(e.startswith("Person") for e in entries):
        break
    subdirs = [e for e in entries if os.path.isdir(os.path.join(DATASET_ROOT, e))]
    if len(subdirs) != 1:
        raise RuntimeError(
            f"couldn't auto-detect DATASET_ROOT under {DATASET_ROOT!r} -- "
            f"found {entries!r}, expected exactly one wrapper folder or Person* folders directly. "
            f"Set DATASET_ROOT by hand and skip this cell's loop."
        )
    DATASET_ROOT = os.path.join(DATASET_ROOT, subdirs[0])
print(f"DATASET_ROOT = {DATASET_ROOT}")

with open("configs/default.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["DATA"]["ROOT"] = DATASET_ROOT
with open("configs/default.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("patched configs/default.yaml DATA.ROOT ->", DATASET_ROOT)

No shareable zip link yet, or prefer the manual route? Mount your own Drive
copy instead (slower per-epoch, no zip/link needed):

```python
from google.colab import drive
drive.mount('/content/drive')
DATASET_ROOT = "/content/drive/MyDrive/dataset"  # wherever you uploaded PersonX/scene_XXXX/
# ... then run the DATA.ROOT-patching cell above with this DATASET_ROOT
```

## 3. Mount Drive (for checkpoint/log persistence)

Separate from the dataset above -- this is so checkpoints AND the loss-history CSV
survive a Colab disconnect (step 6 writes both there).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Sanity check (synthetic data, no dataset needed)

Confirms the 6D-rotation roundtrip, the Hungarian matcher, the set-prediction loss,
and a full model forward/backward pass (on both the spconv and pure-PyTorch backbone
paths already checked locally) all produce finite results before touching the real
dataset -- same role `smoke_test.py` plays for the dense-head experiments.

In [ ]:
!python test_detr_components.py

## 5. Quick batch-size/speed check

Every config's `BATCH_SIZE` (`configs/default.yaml`, inherited by
`configs/exp_detr_head.yaml`) is only measured on an RTX 2070 (Windows) -- Colab GPUs
(T4/A100/etc) are different hardware, so check memory/speed here before committing to
a long run.

In [ ]:
import time

import torch
from torch.utils.data import DataLoader

from config_utils import load_config
from data.dataset import SonarDiverDataset, collate_fn
from models.detector_detr import DiverDetectorDETR

CONFIG_PATH = "configs/exp_detr_head.yaml"
BATCH_SIZE_TO_TEST = None   # None = read OPTIMIZATION.BATCH_SIZE from the config; set an int here to override
N_WARMUP = 5
N_TIMED = 30

device = torch.device("cuda")
cfg = load_config(CONFIG_PATH)
if BATCH_SIZE_TO_TEST is None:
    BATCH_SIZE_TO_TEST = cfg["OPTIMIZATION"]["BATCH_SIZE"]
ds = SonarDiverDataset(cfg, "train")
loader = DataLoader(ds, batch_size=BATCH_SIZE_TO_TEST, shuffle=True, collate_fn=collate_fn, drop_last=True)
model = DiverDetectorDETR(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
steps_per_epoch = len(ds) // BATCH_SIZE_TO_TEST
print(f"train frames: {len(ds)}  steps/epoch at batch={BATCH_SIZE_TO_TEST}: {steps_per_epoch}")

def run_one_step(it):
    try:
        batch = next(it)
    except StopIteration:
        it = iter(loader)
        batch = next(it)
    losses, pred, gt_boxes_list, matches = model.loss(batch, device)
    optimizer.zero_grad()
    losses["total"].backward()
    optimizer.step()
    n_pos = sum(qi.numel() for qi, _ in matches)
    return it, n_pos

it = iter(loader)
torch.cuda.reset_peak_memory_stats()
for _ in range(N_WARMUP):
    it, _ = run_one_step(it)

torch.cuda.synchronize()
t0 = time.perf_counter()
last_n_pos = None
for _ in range(N_TIMED):
    it, last_n_pos = run_one_step(it)
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

sec_per_step = elapsed / N_TIMED
reserved = torch.cuda.max_memory_reserved() / 1024**3
total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
epoch_min = sec_per_step * steps_per_epoch / 60
num_epochs = cfg["OPTIMIZATION"]["NUM_EPOCHS"]

print(f"peak_reserved={reserved:.2f} GiB / {total_mem:.1f} GiB ({100*reserved/total_mem:.0f}%)  "
      f"matched queries(last step)={last_n_pos}")
print(f"{sec_per_step*1000:.0f} ms/step  ->  ~{epoch_min:.1f} min/epoch  ->  "
      f"~{epoch_min*num_epochs/60:.1f} hours for all {num_epochs} epochs (NUM_EPOCHS in the config)")

del model, optimizer, loader, ds
torch.cuda.empty_cache()

## 6. Train

Checkpoints AND the loss-history CSV go to Drive so a disconnect mid-run doesn't lose
progress.

In [ ]:
EXPERIMENT_NAME = "sparsevoxformer_detr"
DRIVE_ROOT = "/content/drive/MyDrive/3d-point-cloud-runs"

!python train_detr.py --config configs/exp_detr_head.yaml \
    --ckpt_dir "{DRIVE_ROOT}/checkpoints/{EXPERIMENT_NAME}" \
    --log_file "{DRIVE_ROOT}/logs/{EXPERIMENT_NAME}/loss_history.csv" \
    --exp_name "{EXPERIMENT_NAME}"

Resume after a disconnect (picks up the schedule/step count from the checkpoint):

In [ ]:
# !python train_detr.py --config configs/exp_detr_head.yaml \
#     --ckpt_dir "{DRIVE_ROOT}/checkpoints/{EXPERIMENT_NAME}" \
#     --log_file "{DRIVE_ROOT}/logs/{EXPERIMENT_NAME}/loss_history.csv" \
#     --exp_name "{EXPERIMENT_NAME}" \
#     --resume "{DRIVE_ROOT}/checkpoints/{EXPERIMENT_NAME}/{EXPERIMENT_NAME}_last.pth"

## 7. Evaluate

`logs/<name>/loss_history.csv` (the Drive path from step 6) has per-step and
per-epoch train/val loss already. For precision/recall/AP-style metrics on the held-out
test split (same style as `test.py`'s dense-head report, so the two heads' numbers are
directly comparable):

In [ ]:
!python test_detr.py --checkpoint "{DRIVE_ROOT}/checkpoints/{EXPERIMENT_NAME}/{EXPERIMENT_NAME}_last.pth" --split test

Add `--pr_curve_out some_path.png` for a PR-curve-per-IoU-threshold plot
(same format as `test.py --pr_curve_out`), or `--per_frame_out some_path.json` for a
per-frame breakdown sorted worst-to-best by F1.